In [ ]:
# import matplotlib

# class mplDeprecation(UserWarning):
#     pass

# matplotlib.cbook.mplDeprecation = mplDeprecation

import matplotlib.pyplot as plt
import seaborn as sns
from textwrap import fill
import numpy as np
import pandas as pd
import scanpy.external as sce
import scanpy as sc
import scvelo as scv
import gget

sc.settings.verbosity = 3 

In [ ]:
scv.__version__

In [ ]:
break

# Load expression

In [ ]:
%%time
fpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/processed.h5ad"
adata = sc.read_h5ad(fpath)
adata.X = adata.layers['log_norm'].copy()
adata

# Load Velocity

In [ ]:
%%time
fpath = "/scratch/indikar_root/indikar1/shared_data/hsc_velocyto/merged_NKUVW.loom"
ldata = scv.read(fpath)
sc.logging.print_memory_usage()
ldata

# Match Cell Ids

In [ ]:
# Set up obs dataframe from ldata
obs = pd.DataFrame({'sv_cell_id': ldata.obs_names})

# Standardize cell ID format to match adata
obs['cell_id'] = obs['sv_cell_id'].apply(lambda x: x.split(":")[1][:-1] + "-1")
obs = obs.set_index('cell_id')
ldata.obs = obs.copy()

# Print initial cell counts
print(f"Initial cell count in ldata: {ldata.shape[0]}")
print(f"Initial cell count in adata: {adata.shape[0]}")

# Filter ldata to keep only cells present in adata
initial_ldata_count = ldata.shape[0]
ldata = ldata[ldata.obs.index.isin(adata.obs_names), :].copy()
print(f"Filtered ldata: {ldata.shape[0]} cells kept, {initial_ldata_count - ldata.shape[0]} cells removed")

# Filter adata to keep only cells present in ldata
initial_adata_count = adata.shape[0]
adata = adata[adata.obs.index.isin(ldata.obs_names), :].copy()
print(f"Filtered adata: {adata.shape[0]} cells kept, {initial_adata_count - adata.shape[0]} cells removed")


# Merge and Filter

In [ ]:
adata = scv.utils.merge(adata, ldata)

# simple filtering
sc.pp.filter_cells(adata, min_genes=500)
sc.pp.filter_genes(adata, min_cells=100)

n_genes = 6000

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=n_genes,
    subset=True,
)

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 7, 5

scv.pl.proportions(adata, groupby='cluster_str', dpi=200)

adata

In [ ]:
sc.pp.pca(adata)
sc.pp.neighbors(
    adata, 
    n_neighbors=15,
)

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 4, 3

sc.pl.pca_variance_ratio(adata)

In [ ]:
scv.pp.filter_and_normalize(adata)

scv.pp.moments(
    adata,
    n_pcs=20,
    n_neighbors=5,
)

adata

In [ ]:
adata.X = adata.X.todense().astype('object')
print(f"{type(adata.X)=}")

In [ ]:
%%time
scv.tl.recover_dynamics(adata)

# scv.tl.velocity(
#     adata,
#     mode='dynamical',
# )

# scv.tl.velocity_graph(
#     adata, 
#     n_jobs=1,
#     n_neighbors=15,
    
# )

In [ ]:
break

In [ ]:
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 4.5, 4.5

scv.pl.velocity_embedding_grid(
    adata, 
    basis='umap',
    color='cluster_str',
)

In [ ]:
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 4.5, 4.5

scv.pl.velocity_embedding_stream(
    adata, 
    basis='umap',
    color='cluster_str',
    title="",
    legend_loc='right_margin',
)

In [ ]:
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 3, 3

scv.tl.velocity_pseudotime(
    adata,
)

scv.pl.scatter(
    adata, 
    color='velocity_pseudotime', 
    cmap='gnuplot',
)

In [ ]:
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 3.5, 2

sns.barplot(
    data=adata.obs,
    x='cluster_str',
    y='velocity_pseudotime',
    hue='cluster_str',
    width=0.5,
    ec='k',
)

sns.despine()

plt.ylabel('pseudotime')
plt.xlabel('')

In [ ]:
scv.tl.latent_time(adata)

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 3, 3

scv.pl.scatter(
    adata, 
    color='latent_time',
    color_map='gnuplot',
    add_outline=True,
)


In [ ]:
n_genes = 25
count_thr = 500
top_genes = adata.var.sort_values(by='fit_likelihood', ascending=False)
top_genes = top_genes[top_genes['n_cells'] > count_thr]

top_genes = top_genes['gene_name'].values[:n_genes]

plt.rcParams['figure.dpi'] = 200

scv.pl.heatmap(
    adata, 
    var_names=top_genes, 
    sortby='latent_time', 
    col_color='cluster_str', 
    figsize=(8, 4.5),
    yticklabels=True,
    n_convolve=30,
)

In [ ]:
n_genes = 15
top_genes = adata.var['fit_likelihood'].sort_values(ascending=False).index

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 2, 3

scv.pl.scatter(
    adata, 
    basis=top_genes[:n_genes], 
    color='cluster_str', 
    ncols=5, 
    frameon=False,
)

In [ ]:
   scv.pl.scatter(
        adata, 
        x='latent_time', 
        y=['CD34', 'ACE'],
        color='cluster_str',
        frameon=False,
        smooth=15,
        dpi=200,
        add_outline=True,
        n_convolve=5,
    )
    plt.show()

In [ ]:
break

In [ ]:
break

In [ ]:
ldata.obs_names[:10]